# Disposability

**Objective:** Start quickly, finish accepted work safely, and release resources during shutdown.

## Simple version

Every opened resource needs a matching cleanup step, including when work is cancelled or fails.

In [ ]:
lifecycle = ["start", "accept work", "stop accepting", "finish work", "close"]

print(" -> ".join(lifecycle))

## Polished version

The worker always closes its resource, stops accepting new work, and drains jobs already accepted before exiting.

In [ ]:
import asyncio
from dataclasses import dataclass


@dataclass
class Resource:
    opened: bool = False

    async def open(self) -> None:
        self.opened = True

    async def close(self) -> None:
        self.opened = False


class GracefulWorker:
    def __init__(self, queue: asyncio.Queue[str], resource: Resource) -> None:
        self.queue = queue
        self.resource = resource
        self.stopping = asyncio.Event()
        self.completed: list[str] = []

    async def run(self) -> None:
        await self.resource.open()
        try:
            while not self.stopping.is_set() or not self.queue.empty():
                try:
                    job = await asyncio.wait_for(self.queue.get(), timeout=0.05)
                except TimeoutError:
                    continue

                try:
                    await asyncio.sleep(0.01)
                    self.completed.append(job)
                finally:
                    self.queue.task_done()
        finally:
            await self.resource.close()

    def stop(self) -> None:
        self.stopping.set()


queue: asyncio.Queue[str] = asyncio.Queue()
for job in ("job-1", "job-2", "job-3"):
    await queue.put(job)

resource = Resource()
worker = GracefulWorker(queue, resource)
running = asyncio.create_task(worker.run())

await asyncio.sleep(0)
worker.stop()
await queue.join()
await running

print("Completed:", worker.completed)
print("Resource open:", resource.opened)

## Applied in this repository

FastAPI lifespan hooks close database, HTTP, and Redis clients. The worker shutdown hook closes provider and Redis connections; queued work has explicit timeouts and retry limits.